# CSOT: 约束分离最优传输相位恢复

## 约束分离 = 像素随机分组 + 相邻像素不能同组 → 非周期、有间隔的最优分布

### 核心思想

**论文1（约束分离方法）**:
- 将SLM像素随机分配到K个组
- **关键约束**: 相邻像素不能属于同一组（图着色约束）
- 效果: 打破周期性，实现非周期、有间隔的最优分布

**论文2（OT相位恢复, arXiv:2408.17025）**:
- 使用Sinkhorn OT + MRAF进行相位恢复
- 所有像素耦合在全局优化中

**CSOT改进**:
1. OT阶段: 按像素组采样 → 空间多样化的随机batch
2. 精化阶段: 分组交替GS投影 → 打破周期性伪影
3. 约束分离: 每组独立更新，相邻像素不同组确保空间去相关

In [ ]:
import sys
sys.path.insert(0, '../../OTPhaseExtractor-main/OTPhaseExtractor-main')

import numpy as np
import matplotlib.pyplot as plt
import torch

from csot_phase import (
    CSOTPhaseRetriever, GroupSeparatedRefiner,
    GroupConstrainedOTRetriever,
    create_pixel_groups, verify_group_constraint,
    create_group_masks, plot_pixel_groups,
    FFT2, IFFT2, fourier_propagate,
    compare_methods, plot_comparison
)
from utils.coordinates import CenteredGridCoordinates
from utils.make_data import make_gauss_cartesian

plt.rcParams['figure.figsize'] = (14, 10)
print('✓ 所有导入成功')

## 1. 约束分离像素分组 — 核心可视化

展示像素如何被分为满足"相邻像素不同组"约束的组。

In [ ]:
shape = (16, 16)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 展示不同分组数
for i, n_groups in enumerate([4, 6, 9]):
    groups = create_pixel_groups(shape, n_groups, seed=42)
    info = verify_group_constraint(groups)
    
    ax = axes[i]
    ax.imshow(groups, cmap='tab10', vmin=0, vmax=n_groups-1)
    
    # 标注每个像素的组号
    for y in range(shape[0]):
        for x in range(shape[1]):
            ax.text(x, y, str(groups[y, x]), ha='center', va='center',
                   fontsize=8, color='white', fontweight='bold')
    
    ax.set_title(f'{n_groups}组 | 违反率: {info["violation_rate"]:.0%}\n'
                f'{"✓ 约束满足" if info["valid"] else "✗ 有违反"}')
    ax.set_xticks(range(shape[1]))
    ax.set_yticks(range(shape[0]))
    ax.grid(True, alpha=0.3)

plt.suptitle('约束分离像素分组: 相邻像素不能同组', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('注意: 4组时(左图), 任何相邻(上下左右)像素都是不同颜色 → 完全满足约束')
print('6组和9组时, 需要更复杂的着色方案, 可能有少量违反')

## 2. 生成测试数据

In [ ]:
N = 128
coords = CenteredGridCoordinates(N, N)
xs, ys = coords.get_mesh()

source = make_gauss_cartesian(xs, ys, 0, 0, N/6)
source /= source.sum()

# 多高斯目标
target = np.zeros_like(source)
for x0, y0, s in [(0,0,N/10), (-N/5,N/5,N/14), (N/5,-N/5,N/14),
                   (-N/5,-N/5,N/16), (N/5,N/5,N/16),
                   (0,N/4,N/20), (0,-N/4,N/20)]:
    target += make_gauss_cartesian(xs, ys, x0, y0, s)
target /= target.sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, data, title in [(axes[0], source, '源 (高斯)'),
                          (axes[1], target, '目标 (多高斯)')]:
    ax.imshow(data, cmap='hot')
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## 3. 对比: GS vs Flat OT vs CSOT

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

results = compare_methods(source, target, xs, ys,
    methods=['gs', 'ot_flat', 'csot'], device=device, seed=42)

print('\n' + '='*60)
print('误差汇总:')
print('='*60)
for method, res in results.items():
    rms = res.get('rms_error',
        np.sqrt(np.mean((res['output_intensity'] - target/target.sum())**2)))
    print(f'  {method:12s}: RMS = {rms:.6f}')

## 4. 结果可视化

In [ ]:
fig, axes = plot_comparison(results, source, target, figsize=(20, 10))
plt.suptitle('相位恢复方法对比 (含像素分组可视化)', fontsize=14, y=1.02)
plt.show()

## 5. 约束分离分组效果分析

对比标准GS（全图同步更新）vs 分组GS（约束分离分组交替更新）的收敛行为。

In [ ]:
# 运行分组约束分离精化 (不依赖OT)
source_amp = np.sqrt(source / source.sum())
target_amp = np.sqrt(target / target.sum())

# 零相位初始化 — 测试纯约束分离精化能力
init_phase = np.zeros_like(source)

# 标准GS (全图同步)
print("运行标准GS (全图同步更新)...")
from extractors.GSphase import GerchbergSaxton2d
gs = GerchbergSaxton2d(num_iter=100)
phase_gs = gs(source, target)

# 分组约束分离GS (分组交替更新)
print("\n运行分组约束分离GS (n_groups=4)...")
refiner = GroupSeparatedRefiner(n_groups=4, n_iter=100, seed=42, verbose=True)
phase_group, groups, errors_group = refiner.refine(source, target, init_phase)

# 可视化对比
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

out_gs = fourier_propagate(source, phase_gs)
out_group = fourier_propagate(source, phase_group)
rms_gs = np.sqrt(np.mean((out_gs - target/target.sum())**2))
rms_group = np.sqrt(np.mean((out_group - target/target.sum())**2))

# 第一行: GS
axes[0,0].imshow(phase_gs, cmap='twilight_shifted')
axes[0,0].set_title(f'GS 相位 (全图同步)')
axes[0,1].imshow(out_gs, cmap='hot')
axes[0,1].set_title(f'GS 输出 (RMS={rms_gs:.4f})')
axes[0,2].text(0.5, 0.5, '所有像素\n同步更新', ha='center', va='center',
              fontsize=14, transform=axes[0,2].transAxes)
axes[0,2].axis('off')

# 第二行: 分组约束分离
axes[1,0].imshow(phase_group, cmap='twilight_shifted')
axes[1,0].set_title(f'分组GS 相位 (n_groups=4)')
axes[1,1].imshow(out_group, cmap='hot')
axes[1,1].set_title(f'分组GS 输出 (RMS={rms_group:.4f})')
plot_pixel_groups(groups, ax=axes[1,2], title='像素分组 (相邻不同组)')

for ax in axes.flat[:4]: ax.axis('off')
plt.suptitle('全图同步 vs 约束分离分组 (零相位初始化, 100次迭代)', fontsize=14)
plt.tight_layout(); plt.show()

print(f'GS (全图同步):   RMS = {rms_gs:.6f}')
print(f'分组GS (4组):    RMS = {rms_group:.6f}')
print(f'改进:            {(1-rms_group/rms_gs)*100:.1f}%')

## 6. 不同分组数的影响

In [ ]:
n_groups_list = [2, 4, 8, 16]
results_by_group = {}

for ng in n_groups_list:
    print(f"\n测试 n_groups={ng}...")
    refiner = GroupSeparatedRefiner(n_groups=ng, n_iter=50,
                                     seed=42, verbose=False)
    phase_g, _, errors_g = refiner.refine(source, target, init_phase)
    results_by_group[ng] = {'phase': phase_g, 'errors': errors_g}

# 绘制收敛曲线
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ng in n_groups_list:
    errs = results_by_group[ng]['errors']
    iters = np.arange(0, len(errs)*10, 10)
    axes[0].plot(iters, errs, 'o-', markersize=3, label=f'n_groups={ng}')
axes[0].set_xlabel('迭代次数'); axes[0].set_ylabel('RMS 误差')
axes[0].set_title('不同分组数的收敛对比 (零相位初始化)')
axes[0].legend(); axes[0].grid(True)

# 最终误差 vs 分组数
final_errs = [results_by_group[ng]['errors'][-1] for ng in n_groups_list]
axes[1].bar(range(len(n_groups_list)), final_errs, tick_label=[str(n) for n in n_groups_list])
axes[1].set_xlabel('分组数'); axes[1].set_ylabel('最终RMS误差')
axes[1].set_title('最终误差 vs 分组数')

plt.tight_layout(); plt.show()

## 算法摘要

### CSOT 算法

```
输入: I_in(x), I_target(x)
输出: φ(x) — SLM相位

# Step 1: 约束分离像素分组
groups = create_pixel_groups(shape, n_groups=4)
# → 4色棋盘格，相邻像素不同组

# Step 2: 分组约束OT初始化 (可选)
for each iteration:
    随机选组 g
    从组g中采样 (空间分散的像素)
    训练对偶势 u_θ, v_ψ
φ_ot = 积分输运映射

# Step 3: 分组约束分离精化
for each iteration:
    随机排列组顺序
    for each group g:
        mask = (groups == g)
        φ_new = GS_one_step(φ_current)  # FFT→振幅替换→IFFT
        φ_current[mask] = φ_new[mask]    # 仅更新本组
```

### 为什么约束分离有效？

1. **打破周期性**: FFT隐含周期性边界条件 → 全图同步更新产生周期性伪影
   → 分组更新打破这种周期性耦合

2. **空间去相关**: 相邻像素不同组 → 每组像素在空间中均匀分散
   → 每组独立优化时看到的是全局长程信息，而非局部短程信息

3. **分块坐标下降**: 每组更新时其他组固定
   → 降低每次更新的耦合维度 → 更容易逃离局部极小值